<div align="center">
  <img src="https://raw.githubusercontent.com/devicons/devicon/master/icons/python/python-original.svg" width="80"/>
</div>

<h1 align="center">Redes Neurais Artificiais, Deep Learning e Algoritmos Genéticos</h1>

<h3 align="center">PhD. Julles Mitoura</h3>

<div align="center">
  <img src="https://img.shields.io/badge/Python-3776AB?style=for-the-badge&logo=python&logoColor=white"/>
  <img src="https://img.shields.io/badge/Jupyter-F37626?style=for-the-badge&logo=jupyter&logoColor=white"/>
  <img src="https://img.shields.io/badge/PyTorch-EE4C2C?style=for-the-badge&logo=pytorch&logoColor=white"/>
</div>

---

## Instalação das dependências

In [ ]:
# Todas as dependências já fazem parte do ambiente padrão
%pip install torch pandas scikit-learn matplotlib --quiet

---

# Aula 4.2: RNN para Análise de Sentimentos

No notebook anterior (Aula 4), aplicamos RNNs a **séries temporais numéricas** — a rede processava uma sequência de valores e aprendia padrões temporais.

Agora vamos aplicar a mesma arquitetura a **texto**, que é naturalmente sequencial: uma frase é uma sequência de palavras, e a ordem importa para o significado.

> *"The movie was not good"* ≠ *"The movie was good"*

O problema que vamos resolver é **análise de sentimentos binária**: dado um texto (review), classificar como **Positivo** ou **Negativo**.

---

## 1. Motivação — Por que RNN para Texto?

Uma **Rede Neural Feed-Forward (FNN)** recebe uma entrada de tamanho fixo. Para texto, isso seria problemático: frases têm tamanhos diferentes e a posição das palavras importa.

A **RNN** resolve isso processando a frase **palavra por palavra**, mantendo um estado oculto $h_t$ que acumula o contexto das palavras anteriores:

$$h_t = \tanh(W_h \cdot h_{t-1} + W_x \cdot x_t + b)$$

Ao final da sequência, $h_T$ (estado oculto da última palavra) contém um resumo da frase inteira — e é esse vetor que usamos para classificação.

```
"The  movie  was  great"
  ↓      ↓     ↓     ↓
 [RNN] [RNN] [RNN] [RNN] → h_T → Linear → Positivo/Negativo
```

---

## 2. Dataset

Vamos usar um dataset de reviews com rótulos **Positive** e **Negative**.

- **Arquivo:** `../03_GenerativeAI/dataset/dataset.csv` (relativo a este notebook)
- **Tamanho:** ~17.000 reviews
- **Classes:** Positive (1) / Negative (0)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('text_data/dataset.csv')

print(f'Shape: {df.shape}')
print(f'\nColunas: {df.columns.tolist()}')
print(f'\nDistribuição de classes:')
print(df['label'].value_counts())
df.head()

In [ ]:
# Visualizar distribuição das classes
contagem = df['label'].value_counts()

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(contagem.index, contagem.values, color=['#2ecc71', '#e74c3c'], edgecolor='black')

for bar, val in zip(bars, contagem.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 100,
            str(val), ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_title('Distribuição das Classes', fontsize=14)
ax.set_xlabel('Sentimento')
ax.set_ylabel('Quantidade')
plt.tight_layout()
plt.show()

In [ ]:
# Mapear rótulos para inteiros
df['label_num'] = df['label'].map({'Positive': 1, 'Negative': 0})

# Comprimento médio das reviews
df['n_words'] = df['text'].apply(lambda t: len(str(t).split()))
print(f'Comprimento médio: {df["n_words"].mean():.1f} palavras')
print(f'Máximo: {df["n_words"].max()} | Mínimo: {df["n_words"].min()}')

---

## 3. Pré-processamento

Para alimentar texto em uma RNN precisamos converter palavras em números. O processo é:

1. **Limpeza** — lowercase, remover pontuação
2. **Vocabulário** — mapear cada palavra a um índice único
3. **Encoding** — transformar cada frase em lista de índices
4. **Padding** — uniformizar o tamanho das sequências com zeros

O token `<PAD>` (índice 0) é usado para completar frases curtas.  
O token `<UNK>` (índice 1) representa palavras fora do vocabulário.

In [ ]:
import re
from collections import Counter

# Hiperparâmetros de pré-processamento
MAX_VOCAB = 10_000   # tamanho máximo do vocabulário
MAX_LEN   = 100      # tamanho máximo da sequência (palavras)

def limpar_texto(texto):
    texto = str(texto).lower()
    texto = re.sub(r'[^a-z\s]', '', texto)   # remove pontuação e números
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

textos  = df['text'].apply(limpar_texto).tolist()
rotulos = df['label_num'].tolist()

print('Exemplo de texto original:')
print(df['text'].iloc[0])
print('\nApós limpeza:')
print(textos[0])

In [ ]:
# Construir vocabulário do zero
contador = Counter(word for texto in textos for word in texto.split())

word2idx = {'<PAD>': 0, '<UNK>': 1}
word2idx.update({
    palavra: idx + 2
    for idx, (palavra, _) in enumerate(contador.most_common(MAX_VOCAB))
})

VOCAB_SIZE = len(word2idx)
print(f'Tamanho do vocabulário: {VOCAB_SIZE}')
print(f'Palavras mais comuns: {contador.most_common(10)}')

In [ ]:
import torch

def encode_e_pad(texto, word2idx, max_len):
    """Converte texto em lista de índices com padding/truncamento."""
    indices = [word2idx.get(word, 1) for word in texto.split()]   # <UNK>=1 para palavras desconhecidas
    indices = indices[:max_len]                                    # truncar se necessário
    indices += [0] * (max_len - len(indices))                     # padding com <PAD>=0
    return indices

X = torch.tensor([encode_e_pad(t, word2idx, MAX_LEN) for t in textos], dtype=torch.long)
y = torch.tensor(rotulos, dtype=torch.float32)

print(f'X shape: {X.shape}')   # (n_amostras, MAX_LEN)
print(f'y shape: {y.shape}')   # (n_amostras,)
print(f'\nExemplo de sequência codificada (primeiros 20 tokens):')
print(X[0, :20].tolist())

In [ ]:
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

BATCH_SIZE = 64

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.1, random_state=42, stratify=y_train
)

loader_treino = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
loader_val    = DataLoader(TensorDataset(X_val,   y_val),   batch_size=BATCH_SIZE)
loader_teste  = DataLoader(TensorDataset(X_test,  y_test),  batch_size=BATCH_SIZE)

print(f'Treino:    {len(X_train)} amostras')
print(f'Validação: {len(X_val)} amostras')
print(f'Teste:     {len(X_test)} amostras')

---

## 4. Modelo RNN Melhorado

A arquitetura combina três aprimoramentos sobre a RNN vanilla:

1. **Embedding + Dropout** — regularização já na camada de embeddings
2. **RNN Bidirecional** — processa a sequência nos dois sentidos, capturando contexto passado e futuro
3. **Mean Pooling** — média de todos os hidden states em vez de usar só o último $h_T$ — mais robusto para frases longas

```
Tokens  →  Embedding  →  Dropout  →  RNN →  →  h_0..T  ↘
                                     RNN ←  ←  h_T..0  →  Mean Pool  →  Linear  →  prob
                               (bidirecional: concatena as duas direções)
```

In [ ]:
import torch.nn as nn

class SentimentRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, n_layers, dropout):
        super().__init__()
        self.embedding  = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.embed_drop = nn.Dropout(dropout)
        self.rnn = nn.RNN(
            input_size=embed_dim,
            hidden_size=hidden_size,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0.0,
            nonlinearity='tanh',
            bidirectional=True          # lê a sequência nos dois sentidos
        )
        self.fc = nn.Linear(hidden_size * 2, 1)   # *2 por causa do bidirecional

    def forward(self, x):
        emb    = self.embed_drop(self.embedding(x))   # (batch, seq_len, embed_dim)
        output, _ = self.rnn(emb)                     # (batch, seq_len, hidden*2)
        out    = output.mean(dim=1)                   # mean pooling sobre os timesteps
        return self.fc(out).squeeze(1)                # (batch,)


# Hiperparâmetros do modelo
EMBED_DIM   = 128
HIDDEN_SIZE = 128   # efetivo = 256 com bidirecional
N_LAYERS    = 2
DROPOUT     = 0.3

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')

modelo = SentimentRNN(VOCAB_SIZE, EMBED_DIM, HIDDEN_SIZE, N_LAYERS, DROPOUT).to(device)
print(modelo)
print(f'\nParâmetros treináveis: {sum(p.numel() for p in modelo.parameters() if p.requires_grad):,}')

---

## 5. Treinamento

Como a saída é um único logit (sem softmax), usamos **BCEWithLogitsLoss** — que combina sigmoid + binary cross-entropy de forma numericamente estável.

O **gradient clipping** (`clip_grad_norm_`) é essencial para RNNs, evitando gradientes explodindo durante o BPTT.

O **ReduceLROnPlateau** reduz a taxa de aprendizado automaticamente quando a validação para de melhorar.

In [ ]:
import torch.optim as optim

EPOCHS   = 15
LR       = 0.001
MAX_NORM = 1.0

criterio   = nn.BCEWithLogitsLoss()
otimizador = optim.Adam(modelo.parameters(), lr=LR)
scheduler  = optim.lr_scheduler.ReduceLROnPlateau(otimizador, mode='min', patience=2, factor=0.5)

historico = {'train_loss': [], 'val_loss': [], 'val_acc': []}

for epoca in range(1, EPOCHS + 1):
    # --- Treino ---
    modelo.train()
    loss_treino = 0.0
    for X_batch, y_batch in loader_treino:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        otimizador.zero_grad()
        logits = modelo(X_batch)
        loss = criterio(logits, y_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(modelo.parameters(), MAX_NORM)
        otimizador.step()
        loss_treino += loss.item()
    loss_treino /= len(loader_treino)

    # --- Validação ---
    modelo.eval()
    loss_val = 0.0
    acertos  = 0
    total    = 0
    with torch.no_grad():
        for X_batch, y_batch in loader_val:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            logits = modelo(X_batch)
            loss_val += criterio(logits, y_batch).item()
            preds = (torch.sigmoid(logits) >= 0.5).float()
            acertos += (preds == y_batch).sum().item()
            total   += y_batch.size(0)
    loss_val /= len(loader_val)
    acc_val   = acertos / total

    scheduler.step(loss_val)
    lr_atual = otimizador.param_groups[0]['lr']

    historico['train_loss'].append(loss_treino)
    historico['val_loss'].append(loss_val)
    historico['val_acc'].append(acc_val)

    print(f'Época {epoca:02d}/{EPOCHS} | '
          f'Loss Treino: {loss_treino:.4f} | '
          f'Loss Val: {loss_val:.4f} | '
          f'Acc Val: {acc_val:.4f} | '
          f'LR: {lr_atual:.5f}')

---

## 6. Resultados

### 6.1 Curvas de Aprendizado

In [ ]:
epocas = range(1, EPOCHS + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Loss
ax1.plot(epocas, historico['train_loss'], label='Treino', marker='o')
ax1.plot(epocas, historico['val_loss'],   label='Validação', marker='s')
ax1.set_title('Loss por Época')
ax1.set_xlabel('Época')
ax1.set_ylabel('BCE Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy
ax2.plot(epocas, historico['val_acc'], label='Validação', color='green', marker='s')
ax2.set_title('Acurácia na Validação')
ax2.set_xlabel('Época')
ax2.set_ylabel('Acurácia')
ax2.set_ylim(0, 1)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 6.2 Acurácia no Conjunto de Teste

In [ ]:
modelo.eval()
acertos = 0
total   = 0

with torch.no_grad():
    for X_batch, y_batch in loader_teste:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        logits = modelo(X_batch)
        preds  = (torch.sigmoid(logits) >= 0.5).float()
        acertos += (preds == y_batch).sum().item()
        total   += y_batch.size(0)

acc_teste = acertos / total
print(f'Acurácia no teste: {acc_teste:.4f} ({acc_teste*100:.2f}%)')

### 6.3 Inferência com Frases Livres

In [ ]:
def prever(texto, modelo, word2idx, max_len, device):
    """Classifica um texto como Positive ou Negative."""
    modelo.eval()
    texto_limpo = re.sub(r'[^a-z\s]', '', texto.lower()).strip()
    indices = encode_e_pad(texto_limpo, word2idx, max_len)
    tensor  = torch.tensor([indices], dtype=torch.long).to(device)

    with torch.no_grad():
        logit = modelo(tensor)
        prob  = torch.sigmoid(logit).item()

    label = 'Positive' if prob >= 0.5 else 'Negative'
    return label, prob


# Exemplos de teste
exemplos = [
    "This movie was absolutely amazing, I loved every second of it!",
    "Terrible experience, complete waste of money and time.",
    "The acting was decent but the plot was confusing.",
    "One of the best films I have ever seen in my life.",
    "I did not enjoy this at all, very disappointing.",
]

print(f'{"Texto":<60} {"Label":<12} {"Prob"}')
print('-' * 80)
for texto in exemplos:
    label, prob = prever(texto, modelo, word2idx, MAX_LEN, device)
    print(f'{texto[:58]:<60} {label:<12} {prob:.4f}')

---

## 7. Próximos Passos

A RNN simples tem uma limitação fundamental: ela **esquece** contextos distantes. Considere a frase:

> *"The movie, despite having a wonderful cast and stunning visuals, was absolutely **terrible**."*

A palavra decisiva (*terrible*) está longe do início, e o estado oculto pode ter perdido esse contexto.

| Modelo | Limitação |
|--------|-----------|
| RNN vanilla | Esquece contextos longos (gradiente que some) |
| **LSTM / GRU** | Célula de memória explícita — resolve o problema |
| **Transformers** | Atenção global — vê toda a sequência de uma vez |

No próximo notebook (**Aula 5**) veremos como a **LSTM** resolve esse problema com suas portas de entrada, esquecimento e saída.